# Sequence Data Construction

This notebook is used to construct the sequence data required for three model schemes:

- Scheme 1 (ObsStep): observation step, event-driven

- Scheme 2 (DailyStep): daily step, time-driven

- Scheme 3 (RF): random forest baseline

In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

# Set a random seed to ensure reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Random seed set to: {RANDOM_SEED}")


Random seed set to: 42


## 1. General Preprocessing

In [22]:
# Load data
data_path = Path('../datasets/data_EUR_reordered.csv')
df = pd.read_csv(data_path)
print(f"Original data shape: {df.shape}")
print(f"Columns: {df.columns}")

Original data shape: (36110, 24)
Columns: Index(['No. of obs', 'Publication', 'control_group', 'sowdur', 'Daily fluxes',
       'crop_class', 'Clay', 'CEC', 'BD', 'pH', 'SOC', 'TN', 'C/N',
       'fertilization_class', 'Split N amount', 'appl_class', 'ferdur', 'Temp',
       'Prec', 'ST', 'WFPS', 'NH4+-N', 'NO3_-N', 'MN'],
      dtype='object')


In [24]:
# Remove features with high missing rates: NH4+-N, NO3_-N, MN (missing rate > 60%)
high_missing_features = ['NH4+-N', 'NO3_-N', 'MN']
df = df.drop(columns=high_missing_features)
print(f"After dropping high missing features: {df.shape}")
print(f"Remaining columns: {df.columns}")

After dropping high missing features: (36110, 21)
Remaining columns: Index(['No. of obs', 'Publication', 'control_group', 'sowdur', 'Daily fluxes',
       'crop_class', 'Clay', 'CEC', 'BD', 'pH', 'SOC', 'TN', 'C/N',
       'fertilization_class', 'Split N amount', 'appl_class', 'ferdur', 'Temp',
       'Prec', 'ST', 'WFPS'],
      dtype='object')


In [25]:
# Handle missing values for C/N and TN (33% missing rate)
# First, forward-fill within each sequence, then fill any remaining missing values with the global median
print(f"C/N missing ratio before: {df['C/N'].isnull().mean():.2%}")
print(f"TN missing ratio before: {df['TN'].isnull().mean():.2%}")

# Group by (Publication, control_group) and forward-fill within each sequence using .ffill()
df['C/N'] = df.groupby(['Publication', 'control_group'])['C/N'].transform(lambda x: x.ffill())
df['TN'] = df.groupby(['Publication', 'control_group'])['TN'].transform(lambda x: x.ffill())

# Fill remaining missing values with the global median
df['C/N'] = df['C/N'].fillna(df['C/N'].median())
df['TN'] = df['TN'].fillna(df['TN'].median())

print(f"C/N missing ratio after: {df['C/N'].isnull().mean():.2%}")
print(f"TN missing ratio after: {df['TN'].isnull().mean():.2%}")


C/N missing ratio before: 33.14%
TN missing ratio before: 33.14%
C/N missing ratio after: 0.00%
TN missing ratio after: 0.00%


In [26]:
# Group by (Publication, control_group) to build sequences
# Sort in ascending order by sowdur
df = df.sort_values(['Publication', 'control_group', 'sowdur']).reset_index(drop=True)

# Count the number of sequences
seq_groups = df.groupby(['Publication', 'control_group'])
n_sequences = len(seq_groups)
print(f"\nTotal number of sequences: {n_sequences}")

# Count the length distribution of sequences
seq_lengths = seq_groups.size()
print("\nSequence length statistics:")
print(seq_lengths.describe())



Total number of sequences: 1524

Sequence length statistics:
count    1524.000000
mean       23.694226
std        22.777601
min         2.000000
25%        12.000000
50%        19.000000
75%        27.000000
max       277.000000
dtype: float64


In [27]:
# Filter out sequences that are too short (<10 steps)
MIN_SEQ_LENGTH = 10
valid_seq_ids = seq_lengths[seq_lengths >= MIN_SEQ_LENGTH].index
df = df[df.set_index(['Publication', 'control_group']).index.isin(valid_seq_ids)].reset_index(drop=True)

# Recount
seq_groups = df.groupby(['Publication', 'control_group'])
n_sequences_filtered = len(seq_groups)
print(f"\nAfter filtering sequences < {MIN_SEQ_LENGTH} steps:")
print(f"Number of sequences: {n_sequences_filtered} (removed {n_sequences - n_sequences_filtered})")
print(f"Total samples: {len(df)}")

seq_lengths_filtered = seq_groups.size()
print("\nFiltered sequence length statistics:")
print(seq_lengths_filtered.describe())



After filtering sequences < 10 steps:
Number of sequences: 1347 (removed 177)
Total samples: 34841

Filtered sequence length statistics:
count    1347.000000
mean       25.865627
std        23.366813
min        10.000000
25%        15.000000
50%        20.000000
75%        28.000000
max       277.000000
dtype: float64


In [28]:
# Data split: randomly split at the sequence level into 90% training set and 10% validation set
all_seq_ids = list(seq_groups.groups.keys())
n_train = int(len(all_seq_ids) * 0.9)
n_val = len(all_seq_ids) - n_train

np.random.shuffle(all_seq_ids)
train_seq_ids = all_seq_ids[:n_train]
val_seq_ids = all_seq_ids[n_train:]

# Create training and validation sets
df['split'] = 'val'
df.loc[df.set_index(['Publication', 'control_group']).index.isin(train_seq_ids), 'split'] = 'train'

print(f"\nData split (random_seed={RANDOM_SEED}):")
print(f"Train sequences: {n_train} ({n_train/len(all_seq_ids):.1%})")
print(f"Val sequences: {n_val} ({n_val/len(all_seq_ids):.1%})")
print(f"Train samples: {(df['split']=='train').sum()}")
print(f"Val samples: {(df['split']=='val').sum()}")



Data split (random_seed=42):
Train sequences: 1212 (90.0%)
Val sequences: 135 (10.0%)
Train samples: 31407
Val samples: 3434


## 2. Scheme 1: Observation Step (ObsStep)

**Characteristics**: Event-driven; each step corresponds to one observation

In [ ]:
def build_obs_step_sequences(df_split: pd.DataFrame) -> list[dict]:
    """
    Construct observation step-length sequence data

    Features:
        - Each step corresponds to one observation
        - Compute the time interval time_delta
        - Handle ferdur: replace -1 with 0
        - Retain all features including ferdur, sowdur, time_delta
    """
    sequences = []

    # Define feature columns
    static_features = ['Clay', 'CEC', 'BD', 'pH', 'SOC', 'TN', 'C/N']
    classification_static_features = ['crop_class']
    dynamic_features = ['Temp', 'Prec', 'ST', 'WFPS']
    # fertilization_features = ['fertilization_class', 'Split N amount', 'appl_class', 'ferdur', 'sowdur']
    target = 'Daily fluxes'

    for (pub, group), seq_df in df_split.groupby(['Publication', 'control_group']):
        seq_df = seq_df.sort_values('sowdur').reset_index(drop=True)

        # Compute the time interval
        time_delta = np.zeros(len(seq_df))
        time_delta[1:] = seq_df['sowdur'].values[1:] - seq_df['sowdur'].values[:-1]

        # Handle ferdur: replace -1 with 0
        ferdur_processed = seq_df['ferdur'].values.copy()
        ferdur_processed[ferdur_processed < 0] = 0

        # Build the sequence dictionary
        sequence = {
            'seq_id': (pub, group),
            'seq_length': len(seq_df),

            # Static Features (Numerical)
            'static_numeric': seq_df[static_features].iloc[0].values.astype(np.float32),

            # Static Features (Classification)
            'static_categorical': {
                feat: seq_df[feat].iloc[0] for feat in classification_static_features
            },

            # Dynamic Features (Numerical)
            'dynamic_numeric': seq_df[dynamic_features].values.astype(np.float32),

            # Fertilization-related features
            'fertilization_categorical': {
                'fertilization_class': seq_df['fertilization_class'].values,
                'appl_class': seq_df['appl_class'].values
            },
            'fertilization_numeric': np.stack([
                seq_df['Split N amount'].values,
                ferdur_processed,
                seq_df['sowdur'].values,
                time_delta
            ], axis=1).astype(np.float32),

            # Target variable
            'target': seq_df[target].values.astype(np.float32)
        }

        sequences.append(sequence)

    return sequences

# Build Training and Validation Sets
print("Building ObsStep sequences...")
train_obs = build_obs_step_sequences(df[df['split'] == 'train'])
val_obs = build_obs_step_sequences(df[df['split'] == 'val'])

print(f"ObsStep Train sequences: {len(train_obs)}")
print(f"ObsStep Val sequences: {len(val_obs)}")
print(f"Example sequence keys: {train_obs[0].keys()}")
print(f"Example static_numeric shape: {train_obs[0]['static_numeric'].shape}")
print(f"Example dynamic_numeric shape: {train_obs[0]['dynamic_numeric'].shape}")
print(f"Example fertilization_numeric shape: {train_obs[0]['fertilization_numeric'].shape}")

Building ObsStep sequences...
ObsStep Train sequences: 1212
ObsStep Val sequences: 135
Example sequence keys: dict_keys(['seq_id', 'seq_length', 'static_numeric', 'static_categorical', 'dynamic_numeric', 'fertilization_categorical', 'fertilization_numeric', 'target'])
Example static_numeric shape: (7,)
Example dynamic_numeric shape: (16, 4)
Example fertilization_numeric shape: (16, 4)


In [31]:
train_obs[0]

{'seq_id': (np.int64(1), np.int64(1)),
 'seq_length': 16,
 'static_numeric': array([28. , 16. ,  1.4,  7.6,  8.1,  0.8, 10.8]),
 'static_categorical': {'crop_class': 'fruit'},
 'dynamic_numeric': array([[18.8,  0. , 23.6, 41. ],
        [17.9,  0. , 22.1, 60. ],
        [19.5,  0. , 22. , 41. ],
        [21.5,  0. , 23.5, 60. ],
        [23.8,  0. , 25.8, 59. ],
        [24.1,  0.8, 21.8, 57. ],
        [22.8,  0. , 23.3, 29. ],
        [26.5,  0.9, 25. , 56. ],
        [24.9,  0. , 29.1, 64. ],
        [17.6,  0. , 21.9, 64. ],
        [21.2,  0. , 18.6, 44. ],
        [18.1,  0. , 19. , 46. ],
        [18.7,  1. , 17.3, 48. ],
        [20. ,  0. , 18.7, 69. ],
        [15.4,  0. , 18.6, 72. ],
        [16.6,  0. , 15.6, 68. ]]),
 'fertilization_categorical': {'fertilization_class': array(['NO', 'NO', 'NO', 'NO', 'NO', 'NO', 'NO', 'NO', 'NO', 'NO', 'NO',
         'NO', 'NO', 'NO', 'NO', 'NO'], dtype=object),
  'appl_class': array(['no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 

In [32]:
# Save ObsStep sequence data
output_dir = Path('../datasets')
output_dir.mkdir(exist_ok=True)

with open(output_dir / 'sequences_obs_step_train.pkl', 'wb') as f:
    pickle.dump(train_obs, f)

with open(output_dir / 'sequences_obs_step_val.pkl', 'wb') as f:
    pickle.dump(val_obs, f)

print(f"\nObsStep sequences saved to {output_dir}")



ObsStep sequences saved to ../datasets


## 3. Schema 2: DailyStep

**Characteristics**: Time-driven, each step corresponds to one day

In [ ]:
def build_daily_step_sequences(df_split: pd.DataFrame) -> list[dict]:
    """
    Construct daily step-sequence data

    Features:
        - Each step corresponds to one day
        - Daily data generated via interpolation
        - `Prec` filled with 0; other numeric features linearly interpolated
        - Fertilization feature reconstruction: Split N amount > 0 on fertilization days, = 0 on other days
        - Remove `ferdur`, `sowdur`
        - Create observation mask
    """
    sequences = []

    # Define feature columns
    static_features = ['Clay', 'CEC', 'BD', 'pH', 'SOC', 'TN', 'C/N']
    classification_static_features = ['crop_class']
    dynamic_numeric_features = ['Temp', 'ST', 'WFPS']  # Linear interpolation required
    prec_feature = 'Prec'  # Pad with 0
    fertilization_categorical = ['fertilization_class', 'appl_class']
    target = 'Daily fluxes'

    for (pub, group), seq_df in df_split.groupby(['Publication', 'control_group']):
        seq_df = seq_df.sort_values('sowdur').reset_index(drop=True)

        # Determine the range of daily step sizes
        min_day = int(seq_df['sowdur'].min())
        max_day = int(seq_df['sowdur'].max())
        n_days = max_day - min_day + 1

        # Create daily indices
        daily_index = np.arange(min_day, max_day + 1)

        # Create observation mask
        observed_mask = np.zeros(n_days, dtype=bool)
        obs_days = seq_df['sowdur'].values.astype(int)
        observed_mask[obs_days - min_day] = True

        # Initialize daily data
        daily_data = {
            'days': daily_index,
            'observed_mask': observed_mask
        }

        # Linear interpolation dynamic numerical features
        for feat in dynamic_numeric_features:
            values = seq_df[feat].values
            interpolated = np.interp(daily_index, obs_days, values)
            daily_data[feat] = interpolated.astype(np.float32)

        # Prec padded with 0
        daily_prec = np.zeros(n_days, dtype=np.float32)
        daily_prec[obs_days - min_day] = seq_df[prec_feature].values.astype(np.float32)
        daily_data[prec_feature] = daily_prec

        # Fertilization feature reconstruction
        # Calculate fertilization dates
        fertilization_days = []
        fertilization_amounts = []

        for _, row in seq_df.iterrows():
            if row['ferdur'] >= 0:  # Fertilization records exist
                fert_day = int(row['sowdur'] - row['ferdur'])
                if fert_day >= min_day and fert_day <= max_day:
                    fertilization_days.append(fert_day)
                    fertilization_amounts.append(row['Split N amount'])

        # Split N amount: Only > 0 on fertilization days
        daily_split_n = np.zeros(n_days, dtype=np.float32)
        for fert_day, amount in zip(fertilization_days, fertilization_amounts, strict=True):
            daily_split_n[fert_day - min_day] = amount
        daily_data['Split N amount'] = daily_split_n

        # fertilization_class, appl_class: Forward fill
        for feat in fertilization_categorical:
            daily_values = np.empty(n_days, dtype=object)
            # Initialize with the first observation value
            daily_values[:] = seq_df[feat].iloc[0]
            # Forward fill
            for _, (day, value) in enumerate(zip(obs_days, seq_df[feat].values, strict=True)):
                day_idx = day - min_day
                daily_values[day_idx:] = value
            daily_data[feat] = daily_values

        # Target variable: Linear interpolation (only true values at observation points)
        target_values = seq_df[target].values
        daily_target = np.interp(daily_index, obs_days, target_values).astype(np.float32)
        daily_data[target] = daily_target

        # Build the sequence dictionary
        sequence = {
            'seq_id': (pub, group),
            'seq_length': n_days,
            'min_day': min_day,
            'max_day': max_day,

            # Static Features (Numerical)
            'static_numeric': seq_df[static_features].iloc[0].values.astype(np.float32),

            # Static Features (Classification)
            'static_categorical': {
                feat: seq_df[feat].iloc[0] for feat in classification_static_features
            },

            # Dynamic Features (Numerical) - Exclude time interval features
            'dynamic_numeric': np.stack([
                daily_data['Temp'],
                daily_data[prec_feature],
                daily_data['ST'],
                daily_data['WFPS']
            ], axis=1),

            # Fertilization-related features
            'fertilization_categorical': {
                feat: daily_data[feat] for feat in fertilization_categorical
            },
            'fertilization_numeric': daily_data['Split N amount'].reshape(-1, 1),

            # Observation mask
            'observed_mask': observed_mask,

            # Target variable
            'target': daily_target
        }

        sequences.append(sequence)

    return sequences

# Build Training and Validation Sets
print("Building DailyStep sequences...")
train_daily = build_daily_step_sequences(df[df['split'] == 'train'])
val_daily = build_daily_step_sequences(df[df['split'] == 'val'])

print(f"DailyStep Train sequences: {len(train_daily)}")
print(f"DailyStep Val sequences: {len(val_daily)}")
print(f"Example sequence keys: {train_daily[0].keys()}")
print(f"Example static_numeric shape: {train_daily[0]['static_numeric'].shape}")
print(f"Example dynamic_numeric shape: {train_daily[0]['dynamic_numeric'].shape}")
print(f"Example fertilization_numeric shape: {train_daily[0]['fertilization_numeric'].shape}")
print(f"Example observed_mask sum: {train_daily[0]['observed_mask'].sum()} / {train_daily[0]['seq_length']}")

Building DailyStep sequences...
DailyStep Train sequences: 1212
DailyStep Val sequences: 135
Example sequence keys: dict_keys(['seq_id', 'seq_length', 'min_day', 'max_day', 'static_numeric', 'static_categorical', 'dynamic_numeric', 'fertilization_categorical', 'fertilization_numeric', 'observed_mask', 'target'])
Example static_numeric shape: (7,)
Example dynamic_numeric shape: (86, 4)
Example fertilization_numeric shape: (86, 1)
Example observed_mask sum: 16 / 86


In [34]:
train_daily[2]

{'seq_id': (np.int64(1), np.int64(4)),
 'seq_length': 86,
 'min_day': 0,
 'max_day': 85,
 'static_numeric': array([28. , 16. ,  1.4,  7.6,  8.1,  0.8, 10.8]),
 'static_categorical': {'crop_class': 'fruit'},
 'dynamic_numeric': array([[18.8       ,  0.        , 23.6       , 39.        ],
        [18.755     ,  0.        , 23.525     , 40.05      ],
        [18.71      ,  0.        , 23.45      , 41.1       ],
        [18.665     ,  0.        , 23.375     , 42.15      ],
        [18.62      ,  0.        , 23.3       , 43.2       ],
        [18.575     ,  0.        , 23.225     , 44.25      ],
        [18.53      ,  0.        , 23.15      , 45.3       ],
        [18.485     ,  0.        , 23.075     , 46.35      ],
        [18.44      ,  0.        , 23.        , 47.4       ],
        [18.395     ,  0.        , 22.925     , 48.45      ],
        [18.35      ,  0.        , 22.85      , 49.5       ],
        [18.305     ,  0.        , 22.775     , 50.55      ],
        [18.26      ,  0.     

check data:

In [35]:
split_n_amount = train_daily[2]['fertilization_numeric']
print('calculated fertilization days: ', np.where(split_n_amount > 0)[0])
print('real fertilization days: ', np.array([20, 27, 34, 41, 48, 55, 62, 69, 76]))


calculated fertilization days:  [20 27 34 41 48 55 62 69 76]
real fertilization days:  [20 27 34 41 48 55 62 69 76]


In [36]:
# Save DailyStep sequence data
with open(output_dir / 'sequences_daily_step_train.pkl', 'wb') as f:
    pickle.dump(train_daily, f)

with open(output_dir / 'sequences_daily_step_val.pkl', 'wb') as f:
    pickle.dump(val_daily, f)

print(f"\nDailyStep sequences saved to {output_dir}")



DailyStep sequences saved to ../datasets


## 4. Schema 3: Random Forest Format

**Characteristics**: Treat each time point as an independent sample

In [39]:
def build_rf_data(df_split: pd.DataFrame) -> pd.DataFrame:
    """
    Build Random Forest format data

    Characteristics:
        - Treat each time point as an independent sample
        - Features: All static features + current time step dynamic features
    """
    # Define id columns
    id_cols = ['No. of obs']
    # Define feature columns
    static_features = ['Clay', 'CEC', 'BD', 'pH', 'SOC', 'TN', 'C/N']
    classification_static_features = ['crop_class']
    dynamic_features = ['Temp', 'Prec', 'ST', 'WFPS']
    fertilization_features = ['fertilization_class', 'Split N amount', 'appl_class', 'ferdur', 'sowdur']
    target = 'Daily fluxes'

    # Copy the data to avoid modifying the original data
    df_rf = df_split.copy()

    # Select all features
    all_features = (
        id_cols + static_features + classification_static_features + dynamic_features + fertilization_features
    )

    # Construct the final dataframe
    rf_data = df_rf[all_features + [target]].copy()

    return rf_data

# Building the Training and Validation Sets
print("Building Random Forest format data...")
train_rf = build_rf_data(df[df['split'] == 'train'])
val_rf = build_rf_data(df[df['split'] == 'val'])

print(f"RF Train samples: {len(train_rf)}")
print(f"RF Val samples: {len(val_rf)}")
print(f"RF features: {train_rf.columns.tolist()}")


Building Random Forest format data...
RF Train samples: 31407
RF Val samples: 3434
RF features: ['No. of obs', 'Clay', 'CEC', 'BD', 'pH', 'SOC', 'TN', 'C/N', 'crop_class', 'Temp', 'Prec', 'ST', 'WFPS', 'fertilization_class', 'Split N amount', 'appl_class', 'ferdur', 'sowdur', 'Daily fluxes']


In [40]:
# Save RF format data
train_rf.to_csv(output_dir / 'rf_data_train.csv', index=False)
val_rf.to_csv(output_dir / 'rf_data_val.csv', index=False)

print(f"\nRF format data saved to {output_dir}")



RF format data saved to ../datasets
